In [2]:
import mlflow
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://localhost:5000")

# Specify the experiment you just created for your LLM application or AI agent.
mlflow.set_experiment("My Application")

# Enable automatic tracing for all OpenAI API calls.
mlflow.openai.autolog()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# The trace of the following is sent to the MLflow server.
client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a helpful weather assistant."},
        {"role": "user", "content": "What's the weather like in Delhi?"},
    ],
)

ChatCompletion(id='chatcmpl-8f0972d5-5fb5-45ff-aa4c-51df3a7b371e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m sorry, but I don’t have real‑time weather data.  \nFor the most up‑to‑date forecast in Delhi, I recommend checking a reliable weather service such as:\n\n- **Weather.com** (AccuWeather)  \n- **BBC Weather**  \n- **The Weather Channel**  \n- **Weather Underground**  \n- Your local news station’s weather app  \n- Google “Delhi weather” (the quick‑info panel that appears in search results)\n\nIf you’re looking for a general idea of Delhi’s climate:\n\n- **Summer (May–June)**: Hot and dry, temperatures often 35\u202f°C–40\u202f°C (95\u202f°F–104\u202f°F). Dusty, with high pollution levels.  \n- **Monsoon (July–September)**: Heavy rain, humidity rises to 70–90\u202f%, temperatures drop slightly to 25–30\u202f°C (77–86\u202f°F).  \n- **Winter (December–January)**: Cooler, temperatures range from 7\u202f°C–20\u202f°C (45\u2

Trace(trace_id=tr-3cfa98ac8890923348205fc77c72036b)

In [1]:
import warnings
warnings.filterwarnings('ignore')
import mlflow
import mlflow.sklearn
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. SETUP: Establish the 'Brain'
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Iris_Final_Comparison")

# 2. DATA: 80/20 Split
iris = datasets.load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# 3. ENABLE AUTOLOG: (Captures Params & Model Files)
mlflow.sklearn.autolog()

def evaluate_and_log(model, X_test, y_test, run_name):
    """Helper function to log all test metrics explicitly"""
    y_pred = model.predict(X_test)
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Explicitly Log to MLflow
    mlflow.log_metric("test_accuracy", acc)
    mlflow.log_metric("test_precision", prec)
    mlflow.log_metric("test_recall", rec)
    mlflow.log_metric("test_f1", f1)
    
    print(f"Logged metrics for {run_name} (Acc: {acc:.4f})")

# --- RUN 1: LOGISTIC REGRESSION ---
with mlflow.start_run(run_name="Logistic_Regression"):
    lr = LogisticRegression(max_iter=200)
    lr.fit(X_train, y_train)
    evaluate_and_log(lr, X_test, y_test, "Logistic")

# --- RUN 2: RANDOM FOREST ---
with mlflow.start_run(run_name="Random_Forest"):
    rf = RandomForestClassifier(n_estimators=100)
    rf.fit(X_train, y_train)
    evaluate_and_log(rf, X_test, y_test, "Random Forest")

2026/07/28 20:38:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/28 20:38:13 INFO mlflow.store.db.utils: Updating database tables
2026/07/28 20:38:16 INFO mlflow.tracking.fluent: Experiment with name 'Iris_Final_Comparison' does not exist. Creating a new experiment.
2026/07/28 20:38:32 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - w

Logged metrics for Logistic (Acc: 1.0000)


2026/07/28 20:39:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged metrics for Random Forest (Acc: 1.0000)
